# Automatic Report Evaluation with LLM-as-a-judge

In [ ]:
import pandas as pd
from openai import OpenAI
import json
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv("../.env")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
client = OpenAI()

In [1]:
def get_prompt(report_baseline, report_final):
  prompt = f"""
  You are an expert academic report evaluator. Your task is to compare two reports and give both reports scores according to two groups of indicators: Content Quality and Hallucination Detection.

  Instructions:
  - For each report, score each indicator on a continuous scale from 0 to 1, where 0 = very poor and 1 = excellent.
  - Provide a brief justification for each LLM-evaluated score.
  - Return only valid JSON with the following structure:

  {{
    "report_1":{{
    "content_quality": {{
      "convincing": {{"score": X, "comment": "..."}},
      "balance": {{"score": X, "comment": "..."}},
      "usefulness": {{"score": X, "comment": "..."}},
      "informativeness": {{"score": Y, "comment": "..."}},
      "coherence": {{"score": X, "comment": "..."}}
    }},
    "hallucination_detection": {{
      "economic_validity": {{"score": X, "comment": "..."}},
      "correctness_of_conclusions": {{"score": X, "comment": "..."}},
      "fabrication_check": {{"score": X, "comment": "..."}},
      "emotional_neutrality": {{"score": X, "comment": "..."}},
      "attention_to_key_information": {{"score": X, "comment": "..."}}
    }}
    }}
    "report_2":{{
    "content_quality": {{
      ...
    }},
    "hallucination_detection": {{
      ...
    }}
    }}
  }}
  
  **Indicators Description:**

  Content Quality Indicators:
  1. Convincingness: Measures how the report uses numeric facts or frequency terms to support claims. Higher scores reflect evidence clearly linked to conclusions.
  2. Balance of Key Factors: Evaluates whether the report considers all three relevant factors equally, without overemphasis or omission.
  3. Usefulness for Prediction: Assesses whether the report includes clearly stated predictions or forward-looking implications based on the data.
  4. Informativeness: Checks whether the content is understandable for the target academic level. It should be targetted for educated people.
  5. Coherence & Consistency: Evaluates whether conclusions logically follow from the arguments and remain consistent throughout the report.

  Hallucination Detection Indicators:
  1. Economic Validity: Determines whether statements align with basic economic logic and do not violate well-known theory or market mechanisms.
  2. Correctness of Conclusions: Checks whether the conclusions are logically and empirically correct rather than misleading or unjustified.
  3. Fabrication Check: Identifies whether the report invents data, sources, theories, or claims that are not supported by the text or real-world evidence.
  4. Emotional Neutrality: Examines whether emotional language or sentiment is used to manipulate perception, rather than presenting objective analysis.
  5. Attention to Key Information: Measures whether critical or high-value information receives appropriate attention rather than being ignored or downplayed.

  Report 1 text: {report_baseline}
  Report 2 text: {report_final}
  """
  return prompt


In [ ]:
def get_machine_evaluation(report_baseline, report_final):
    prompt=get_prompt(report_baseline, report_final)
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    scores = json.loads(response.choices[0].message.content)
    print(f"\nbaseline:\n{report_baseline}")
    print(f"\nfinal:\n{report_final}")
    print(scores)
    return scores

## LLM Evaluation

In [ ]:
evaluation_df = pd.read_csv("reports-machine-eval.csv", sep="\t")

In [ ]:
results = []

v0_reports = evaluation_df["v0"].dropna()
v_final_reports = evaluation_df["v_all"].dropna()
company_names = evaluation_df["company"].dropna()

In [7]:
for report_baseline,report_final in zip(v0_reports,v_final_reports):
    scores = get_machine_evaluation(report_baseline,report_final)
    results.append(scores)


baseline:
### Key Rating Drivers for CoreCivic, Inc. (2025)

    #### F1 - Leveraged Capital Structure and Limited Financial Flexibility  
    CoreCivic maintains a relatively high leverage ratio, with significant debt outstanding that pressures its credit metrics. The company's debt maturity profile includes sizable near-term repayments, which could strain liquidity in a stressed operating environment. Additionally, CoreCivic’s interest coverage ratios have shown volatility, reflecting sensitivity to operating cash flow fluctuations. This constrains financial flexibility and limits the ability to absorb unexpected adverse events or invest in growth initiatives without further increasing leverage or diluting equity.

    #### F1 - Profitability Pressures Amid Operational Challenges  
    Profit margins have compressed due to rising operational costs, including labor and compliance expenses. Although CoreCivic benefits from long-term contracts with government entities, variations in oc

In [ ]:
def flatten_evaluation(ev):
    flat = {}
    if not isinstance(ev, dict):
        return flat
    for group in ("content_quality", "hallucination_detection"):
        group_val = ev.get(group, {}) or {}
        for ind, val in group_val.items():
            key = f"{group}.{ind}"
            if isinstance(val, dict):
                flat[key] = {"score": val.get("score"), "comment": val.get("comment")}
            else:
                flat[key] = {"score": None, "comment": None}
    return flat

records = []

v0_results = [result["report_1"] for result in results]
v_final_results = [result["report_2"] for result in results]

for company, ev0, evf in zip(company_names, v0_results, v_final_results):
    flat0 = flatten_evaluation(ev0)
    flatf = flatten_evaluation(evf)
    all_keys = sorted(set(flat0.keys()) | set(flatf.keys()))
    for key in all_keys:
        records.append({
            "company": company,
            "indicator": key,
            "v0_score": flat0.get(key, {}).get("score"),
            "v0_comment": flat0.get(key, {}).get("comment"),
            "v_final_score": flatf.get(key, {}).get("score"),
            "v_final_comment": flatf.get(key, {}).get("comment"),
        })

df_llm_eval = pd.DataFrame.from_records(records).set_index(["company", "indicator"])
df_llm_eval.to_excel("machine_eval_results.xlsx")
df_llm_eval

v0_score  \
company              indicator                                                      
Core Civics          content_quality.balance                                 0.80   
                     content_quality.coherence                               0.80   
                     content_quality.convincing                              0.45   
                     content_quality.informativeness                         0.60   
                     content_quality.usefulness                              0.50   
                     hallucination_detection.attention_to_key_inform...      0.60   
                     hallucination_detection.correctness_of_conclusions      0.60   
                     hallucination_detection.economic_validity               0.85   
                     hallucination_detection.emotional_neutrality            0.95   
                     hallucination_detection.fabrication_check               0.80   
Disney               content_quality.balance                                 0.70   
                     content_quality.coherence                               0.80   
                     content_quality.convincing                              0.45   
                     content_quality.informativeness                         0.65   
                     content_quality.usefulness                              0.40   
                     hallucination_detection.attention_to_key_inform...      0.50   
                     hallucination_detection.correctness_of_conclusions      0.80   
                     hallucination_detection.economic_validity               0.90   
                     hallucination_detection.emotional_neutrality            0.95   
                     hallucination_detection.fabrication_check               0.90   
Alaska Air           content_quality.balance                                 0.80   
                     content_quality.coherence                               0.80   
                     content_quality.convincing                              0.40   
                     content_quality.informativeness                         0.60   
                     content_quality.usefulness                              0.40   
                     hallucination_detection.attention_to_key_inform...      0.75   
                     hallucination_detection.correctness_of_conclusions      0.85   
                     hallucination_detection.economic_validity               0.90   
                     hallucination_detection.emotional_neutrality            0.95   
                     hallucination_detection.fabrication_check               0.90   
Merck and Co         content_quality.balance                                 0.70   
                     content_quality.coherence                               0.80   
                     content_quality.convincing                              0.60   
                     content_quality.informativeness                         0.70   
                     content_quality.usefulness                              0.50   
                     hallucination_detection.attention_to_key_inform...      0.70   
                     hallucination_detection.correctness_of_conclusions      0.75   
                     hallucination_detection.economic_validity               0.90   
                     hallucination_detection.emotional_neutrality            0.95   
                     hallucination_detection.fabrication_check               0.85   
Occidental Petroleum content_quality.balance                                 0.70   
                     content_quality.coherence                               0.80   
                     content_quality.convincing                              0.35   
                     content_quality.informativeness                         0.65   
                     content_quality.usefulness                              0.45   
                     hallucination_detection.attention_to_key_inform...      0